In [1]:
import pandas as pd
import numpy as np
import commands
from data_main import Data
from shapely.geometry import Point, LineString
import math
import helper

In [2]:
args = commands.get_commands()[2]
data = Data(args=args)
data.read_data()

In [12]:
def demand_deterministic_landfall(data, track_err, hurr_cat, t):
    """Compute demand for the given Hurricane position, category,
    and the time period in deterministic landfall case.

    hurr_pos : tuple
        (long, lat) as the position of hurricane track.
    hurr_cat: int
        Hurricane category = 0, 1, 2, 3, 4, or 5.
    t : int
        time period = 0, 1, ...,9, or 10.
    """
    DF = np.zeros([data.I])
    # First demand is realized 4 periods before landfall.
    if t >= data.T - data.tau:
        dY = [0] * data.I
        dX = [0] * data.I
        for i in range(data.I):
            hurr_pos =  [data.FORE.X[t] + data.X_rotate * track_err/data.Xmiles,
                         data.FORE.Y[t] + data.Y_rotate * track_err/data.Ymiles]
            proj_point = data.study_line.interpolate(
                data.study_line.project(Point(hurr_pos))
                )
            dY[i] = math.sqrt(
                ((proj_point.x - data.DP_GIS[i, 0]) * data.Xmiles)**2 +
                ((proj_point.y - data.DP_GIS[i, 1]) * data.Ymiles)**2
                )
            dX[i] = math.sqrt(
                ((proj_point.x - hurr_pos[0]) * data.Xmiles)**2 +
                ((proj_point.y - hurr_pos[1]) * data.Ymiles)**2
                )
            if data.demand_zone.contains(Point(hurr_pos)):
                DF_by_int = helper.demandByWS(hurr_cat)
                DF_by_time = 1 - dX[i] / data.x_max
                DF_by_track = 1 - dY[i] / data.study_region_dist
                DF[i] = DF_by_track * DF_by_time * DF_by_int
    return DF.round(3).tolist()

# Now call the demand function for testing
DF = demand_deterministic_landfall(data, track_err=0, hurr_cat=2, t=5)
# Print full demand list
print("Full demand:", DF)

Full demand: [0.038, 0.044, 0.033, 0.032, 0.032, 0.03, 0.026]


In [ ]:
#np.random.seed(42)
def in_sample(t, state, S=100):
    dim = len(state)
    in_sample = np.zeros([S, data.T, dim])
    for s in range(S):
        sampled_state = state
        in_sample[s, t, :] = state
        for t_ in range(t + 1, data.T):
            a = range(len(data.state_space[t_]))
            p = list(data.pi_mssp[t_ - 1][sampled_state].values())
            sampled_index = np.random.choice(a=a, p=p,)
            sampled_state = data.state_space[t_][sampled_index]
            in_sample[s, t_, :] = sampled_state
    return in_sample

t = 5
initial_state = data.state_space[t][0]

sample1 = in_sample(t, initial_state, S=1)
sample2 = in_sample(t, initial_state, S=1)

print("Sample path 1, time 6:", sample1[0, 6])
print("Sample path 2, time 6:", sample2[0, 6])


Sample path 1, time 6: [-251.49    1.  ]
Sample path 2, time 6: [-251.49    0.  ]


In [39]:
t = 5  # any time t < data.T
initial_state = data.state_space[t][0]  # pick the first state at time t
sample_paths = in_sample(t, initial_state, S=3)

print("Shape of sample_paths:", sample_paths.shape)
print("First sample path:")
print(sample_paths[0])  # prints a (T, dim) array

Shape of sample_paths: (3, 11, 2)
First sample path:
[[ 0.00000e+00  0.00000e+00]
 [ 0.00000e+00  0.00000e+00]
 [ 0.00000e+00  0.00000e+00]
 [ 0.00000e+00  0.00000e+00]
 [ 0.00000e+00  0.00000e+00]
 [-1.45970e+02  0.00000e+00]
 [-2.51490e+02  0.00000e+00]
 [-3.36840e+02  1.00000e+00]
 [-5.26810e+02  2.00000e+00]
 [-1.11062e+03  1.00000e+00]
 [-1.77865e+03  1.00000e+00]]


In [31]:
print("Sample path 1, time 5:", sample_paths[0, 5])  # state at t=5
print("Sample path 1, time 6:", sample_paths[0, 6])  # state at t=6
print("Sample path 2, time 7:", sample_paths[1, 7])  # another sample


Sample path 1, time 5: [-145.97    0.  ]
Sample path 1, time 6: [-251.49    1.  ]
Sample path 2, time 7: [-336.84    1.  ]
